# EIT Numerical Analysis — Maxwell-Bloch Simulation

Simulates **Electromagnetically Induced Transparency (EIT)** in a 1-D atomic
medium by time-marching the coupled Maxwell-Bloch equations:

| Field | Equation |
|-------|----------|
| Probe field E | ∂E/∂t = Ng·P − c·∂E/∂z |
| Polarization P | ∂P/∂t = −γ₁·P + Ng·E + iΩ·S |
| Spin wave S | ∂S/∂t = −γ₂·S + iΩ\*·P |

Integration uses **4th-order Runge-Kutta** with a **4th-order finite-difference**
spatial gradient, JIT-compiled via **Numba** for speed.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.animation as animation
import time
import warnings

# numba ≥ 0.61 required for Python 3.13 support.
# Install: pip install "numba>=0.61"
from numba import njit, prange

warnings.filterwarnings('ignore')

# Render animations inline as HTML (works in classic Notebook and JupyterLab)
plt.rcParams["animation.html"] = "jshtml"

# Set interactive=True only when %matplotlib widget (ipympl) is active
interactive = False
blit = interactive


## Physics helper functions


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Complex decay rates (absorb detunings into imaginary parts)
# ─────────────────────────────────────────────────────────────────────────────

def gamma_1(gamma, delta_1, delta_2):
    """
    Complex decay rate for the optical coherence ρ_eg.
        γ₁ = γ_eg − i(Δ₁ + Δ₂)
    Δ₁: one-photon detuning; Δ₂: two-photon detuning.
    """
    return gamma - 1j * (delta_1 + delta_2)


def gamma_2(gamma0, delta_2):
    """
    Complex decay rate for the spin-wave coherence ρ_sg.
        γ₂ = γ_sg − i·Δ₂
    """
    return gamma0 - 1j * delta_2


def group_velocity(c, Rabi, Ng):
    """
    EIT slow-light group velocity.
        vg = c / (1 + |Ng|² / |Ω|²)   ≈  c·|Ω|² / |Ng|²  in the deep-EIT limit.
    Returns 0 when the control field Ω is off.
    """
    if Rabi == 0:
        return 0.0 + 0.0j
    return c / (1 + (abs(Ng) / abs(Rabi)) ** 2)


def susceptibility(w, delta_2, gamma, gamma0, Ng, Rabi, k1, delta_1=0):
    """
    EIT susceptibility χ(ω).
    Im(χ) ∝ absorption;  Re(χ) ∝ dispersion (→ slow light).
    k1 is a coupling pre-factor proportional to atom density.
    """
    g1 = gamma  - 1j * (delta_1 + delta_2)   # γ₁  (complex)
    g2 = gamma0 - 1j * delta_2                # γ₂  (complex)
    denom = (g1 - 1j * w) - (1j * Rabi) * (1j * np.conjugate(Rabi)) / (g2 - 1j * w)
    return k1 / denom


# Vectorised versions for frequency-sweep plots
susceptibility_v = np.vectorize(susceptibility)
group_velocity_v  = np.vectorize(group_velocity)


## Simulation Parameters

All physical quantities are in **natural EIT units**:
* Length → mm  
* Time   → ns  
* Frequency / rate → GHz  
* Speed of light c = 0.3 mm/ns

Edit the block below to explore different regimes.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Physical parameters  (all quantities in mm / ns / GHz)
# ─────────────────────────────────────────────────────────────────────────────

c  = 0.3      # speed of light [mm/ns]
g  = 5       # atom-photon coupling constant [GHz]
N  = 100      # number of atoms in the ensemble

# ── Detunings ─────────────────────────────────────────────────────────────────
# delta_1 (Δ): one-photon detuning (probe from |g⟩→|e⟩ resonance) [GHz]
#   0  → on-resonance (strongest EIT effect)
#   ≠0 → off-axis; shifts EIT window; P grows
delta_1 = 0.0

# delta_2 (δ): two-photon detuning (probe − control from two-photon resonance) [GHz]
#   0  → perfect dark state; P and absorption are suppressed
#   ≠0 → breaks dark state; P grows; storage degrades
delta_2 = 0.0

# ── Decay / dephasing rates [GHz] ─────────────────────────────────────────────
decay_e   = 50 / 3   # spontaneous emission rate of |e⟩ (natural linewidth)
dephase_s = 5e-4     # spin-wave (ground-state) dephasing rate
dephase_e = 0.0      # additional pure dephasing of |e⟩

# Half-linewidth decay rates for each coherence
gamma    = gamma_eg = 0.5 * (decay_e + dephase_e)    # ρ_eg decay rate
gamma0   = gamma_sg = 0.5 * dephase_s                # ρ_sg decay rate
gamma_es = 0.5 * (decay_e + dephase_e + dephase_s)

# Complex decay rates (absorb detunings for convenience in the ODE)
gamma1 = gamma_1(gamma, delta_1, delta_2)
gamma2 = gamma_2(gamma0, delta_2)

# ── Grid and time step ─────────────────────────────────────────────────────────
dz = 0.004   # spatial step [mm]

# !! STABILITY NOTE !!
# The correct criterion for this wave equation (∂E/∂t + c·∂E/∂z = ...) is the
# CFL (Courant–Friedrichs–Lewy) number:
#
#     CFL = c · dt / dz   must be < 1
#
# With c = 0.3 mm/ns and dz = 0.004 mm:
#     dt = 0.005  →  CFL = 0.375   ✓  stable
#     dt = 0.0005 →  CFL = 0.0375  ✓  stable but 10× more steps than needed
#     dt = 0.014  →  CFL = 1.05    ✗  unstable
#
# Do NOT use dz/dt as the stability metric — it has no physical meaning here.
dt = 0.005   # time step [ns]  →  CFL = c·dt/dz = 0.375  (safe, efficient)

medium_length = 20.0     # EIT medium length [mm]
storage_time  = 1000.0   # hold excitation as a spin wave for this long [ns]

max_amp_rabi  = 30.0    # peak Rabi frequency of control field [GHz]

print(f"γ_eg = {gamma:.4f} GHz   γ_sg = {gamma0:.2e} GHz")
print(f"CFL  = c·dt/dz = {c*dt/dz:.3f}  ({'✓ stable' if c*dt/dz < 1 else '✗ UNSTABLE — reduce dt'})")


## Guide: manipulating the P-field through input parameters

The polarization P obeys  **∂P/∂t = −γ₁·P + Ng·E + iΩ·S**  with  γ₁ = γ_eg − i(Δ₁+Δ₂).

| Parameter | Variable | Effect on P |
|-----------|----------|-------------|
| Excited-state decay | `decay_e` → `gamma` | Sets the linewidth. Larger γ_eg → stronger P damping, broader EIT transparency window. |
| Ground-state dephasing | `dephase_s` → `gamma0` | Larger γ_sg → S decays faster → less iΩ·S feedback → P rises (EIT degrades). Long memory requires γ_sg → 0. |
| One-photon detuning | `delta_1` | Shifts Im(γ₁). For |Δ₁| >> γ_eg, absorption is off-peak and P grows; the signal is phase-shifted but less absorbed. |
| Two-photon detuning | `delta_2` | Breaks the EIT dark state. Any δ ≠ 0 causes P to ring at frequency δ during propagation (dark-state precession). |
| Control Rabi frequency | `max_amp_rabi` | **Main EIT knob.** Larger Ω → stronger P↔S mixing → deeper EIT (P suppressed). Setting Ω = 0 removes EIT entirely; P responds directly to E. |
| Coupling strength | `g`, `N` → `Ng = ig√N` | Larger Ng → E drives P more strongly → higher optical depth OD ∝ \|Ng\|². Also sets the compression of the pulse inside the medium. |
| Switching sharpness | `sharpness` | Faster switching → non-adiabatic; P spikes at each switch edge. Slow switching (small sharpness) preserves dark-state condition. |
| Storage duration | `storage_time` | Longer storage → more γ_sg·S decay → S delivers less to P during retrieval → lower efficiency. |

### Quick recipes

| Goal | Parameter changes |
|------|------------------|
| **Fully suppress P** (ideal EIT) | δ = 0, Δ = 0, Ω >> γ_eg, dephase_s ≈ 0 |
| **Maximise P** (absorbing medium) | Ω = 0; P tracks E directly |
| **Oscillating P pattern** (dark-state beating) | δ ≠ 0 (e.g. 0.01 GHz); P rings at frequency δ |
| **Slow P relaxation** (long memory) | dephase_s → very small, large storage_time |
| **Widen EIT window** (broader P suppression) | Increase max_amp_rabi |
| **Maximise optical depth** | Increase g or N |


## Signal and control-field pulse shapes


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Pulse-shaping helpers
# ─────────────────────────────────────────────────────────────────────────────

def time_pulse(storage_time, start_time, ts, sharpness):
    """
    Temporal Rabi envelope Ω(t): high → ramp-off (store) → ramp-on (retrieve).

    The control field is:
      • HIGH  before storage: compresses the probe into the medium
      • OFF   during storage: traps excitation as a spin wave S
      • HIGH  after  storage: retrieves S back into E

    Parameters
    ----------
    storage_time : float  — off-window duration [same units as ts]
    start_time   : float  — time when the ramp-off begins
    sharpness    : float  — switching steepness (apply 10^(-1/(s+0.3)) rescaling first)

    Returns
    -------
    1-D array, same shape as ts, with minimum shifted to 0.
    """
    offset1       = np.arctanh(0.8) + start_time
    storage_start = offset1 - np.arctanh(-0.8)
    offset2       = storage_start + storage_time - np.arctanh(-0.8)

    edge_off = (np.tanh(-(ts - offset1) * sharpness) + 1) / 2   # falling edge
    edge_on  = (np.tanh( (ts - offset2) * sharpness) + 1) / 2   # rising edge
    Rabi     = edge_off + edge_on
    return Rabi - np.min(Rabi)


## Signal (probe) pulse

Choose `pulse_type = 'gaussian'` or `'square'` below.


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Probe pulse  — set pulse_type to switch between shapes
# ─────────────────────────────────────────────────────────────────────────────
pulse_type = 'gaussian'   # 'gaussian'  or  'square'

t_FWHM = 2                            # Gaussian FWHM in time [ns]
tau    = t_FWHM / 2.355               # 1/e half-width in time  [ns]
sigma  = tau * c                      # 1/e half-width in space [mm]
lim    = sigma * 4 * 2.355            # grid padding on each side of the pulse [mm]

pulse_width_outside = 2 * lim         # total spatial extent of the pulse [mm]
enter  = lim                          # z where the medium begins [mm]
exit_  = enter + medium_length        # z where the medium ends   [mm]

# Grid: left padding = lim, medium = medium_length, right padding = lim only.
# Right side needs just one lim because the retrieved pulse exits slowly; using
# lim*4 (previous default) bloated the grid 2-3x unnecessarily.
z_real = np.arange(-lim, medium_length + lim * 2, dz)
z      = z_real.astype(np.complex128)

if pulse_type == 'gaussian':
    amplitude = 5 / (sigma * (2 * np.pi) ** 0.5)
    signal    = amplitude * np.exp(-z_real**2 / (2 * sigma**2))
    label     = 'Gaussian pulse'
else:
    signal    = np.zeros(len(z_real))
    i_lo      = np.argmin(np.abs(z_real + 2))
    i_hi      = np.argmin(np.abs(z_real - 2))
    signal[i_lo:i_hi] = 1.0
    label     = 'Square pulse'

signal = signal.astype(np.complex128)

plt.figure(figsize=(9, 3))
plt.plot(z_real, signal.real, color='steelblue', label=label)
plt.axvline(enter, color='crimson', linestyle='--', lw=1.2, label='Medium start')
plt.axvline(exit_, color='crimson', linestyle=':',  lw=1.2, label='Medium end')
plt.xlabel('z  [mm]')
plt.ylabel('Amplitude  [arb.]')
plt.title(f'Initial probe pulse — {label}')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

print(f"Grid points: {len(z)}  |  z range: [{z_real.min():.2f}, {z_real.max():.2f}] mm")


## Rabi control field — spatial and temporal profiles


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Atom-density coupling: Ng = i·g·√N  (imaginary so Ng·E is π/2 out of phase)
#   Non-zero only inside the medium; zero outside.
# ─────────────────────────────────────────────────────────────────────────────
Ng = 1j * g * N**0.5 * np.ones(len(z), dtype=np.complex128)

enter_idx = np.argmin(np.abs(z_real - enter))
exit_idx  = np.argmin(np.abs(z_real - exit_))
Ng[:enter_idx] = 0
Ng[exit_idx:]  = 0

# Group velocity at peak Ω (for timing estimates; propagation uses c)
vg_peak    = group_velocity(c, max_amp_rabi, 1j * g * N**0.5)
cfl        = c * dt / dz   # CFL number — must be < 1

pulse_width_inside = vg_peak * (pulse_width_outside / c)
start_time = float(
    ((pulse_width_outside / c) + (medium_length / 2 - pulse_width_inside) / vg_peak).real
)
extra_time = float((medium_length / (2 * vg_peak)).real)
run_time   = 1.5 * storage_time + start_time + extra_time

print(f"Slow-down factor  : {float((c / vg_peak).real):.1f}×")
print(f"start_time        : {start_time:.2f} ns")
print(f"run_time          : {run_time:.2f} ns")
print(f"Expected steps    : {int(run_time / dt):,}   (= run_time / dt = {run_time:.1f} / {dt})")
print(f"CFL               : {cfl:.3f}   ({'✓ stable' if cfl < 1 else '✗ UNSTABLE'})")
print()
if cfl > 0.9:
    print("⚠  CFL is close to 1 — consider reducing dt slightly.")
if cfl < 0.05:
    print(f"⚠  CFL = {cfl:.3f} is very small — dt may be unnecessarily tiny.")
    print(f"   Try dt = {dz / c * 0.4:.4f} ns  (CFL ≈ 0.4)  for a 10× speedup.")

# Temporal Rabi profile
sharpness    = 0.1   # higher → sharper switching; lower → smoother ramp
sharpness    = 10 ** (-1 / (sharpness + 0.3))

tlist        = np.arange(0, run_time, dt)
rabi_profile = max_amp_rabi * time_pulse(storage_time, start_time, tlist, sharpness)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(tlist, rabi_profile, color='darkorange', lw=1.5)
axes[0].set_xlabel('time  [ns]');  axes[0].set_ylabel('Ω(t)  [GHz]')
axes[0].set_title('Control-field Rabi profile Ω(t)')

axes[1].plot(z_real, np.abs(Ng), color='steelblue', lw=1.5)
axes[1].axvline(enter, color='crimson', linestyle='--', label='Medium boundary')
axes[1].axvline(exit_, color='crimson', linestyle='--')
axes[1].set_xlabel('z  [mm]');  axes[1].set_ylabel('|Ng|  [GHz]')
axes[1].set_title('Atom-density coupling profile |Ng(z)|')
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()


## JIT-compiled RK4 integrator

The three fields (E, P, S) are advanced by one time step using
**classical 4th-order Runge-Kutta**.  Each right-hand-side function
maps directly to one line of the Maxwell-Bloch equations.

`dE`, `dP`, `dS` are the accumulated stage increments (0 on the first stage).


In [ ]:
@njit(fastmath=True, parallel=True)
def grad_jit(f, dz):
    """
    4th-order centred finite-difference gradient with 2nd-order boundary stencils.
    parallel=True enables multi-core execution of the interior prange loop.
    """
    g = np.zeros_like(f)
    n = len(f) - 1
    for i in prange(2, n - 1):
        g[i] = (-f[i+2] + 8*f[i+1] - 8*f[i-1] + f[i-2]) / (12 * dz)
    # Left boundary (forward 2nd-order stencil)
    g[0] = (-3*f[0] + 4*f[1] - f[2]) / (2 * dz)
    g[1] = ( f[2]   - f[0])           / (2 * dz)
    # Right boundary (backward 2nd-order stencil)
    g[n]   = (3*f[n]  - 4*f[n-1] + f[n-2]) / (2 * dz)
    g[n-1] = (f[n]    - f[n-2])             / (2 * dz)
    return g


# ── Maxwell-Bloch right-hand sides ───────────────────────────────────────────
# dE / dP / dS are RK stage increments (scalar 0 on stage 1, arrays on 2–4).

@njit(fastmath=True)
def rhs_E(E, dE, P, dP, Ng, c_light, dz):
    """∂E/∂t = Ng·(P+dP) − c·∂(E+dE)/∂z"""
    return Ng * (P + dP) - c_light * grad_jit(E + dE, dz)

@njit(fastmath=True)
def rhs_P(E, dE, P, dP, S, dS, gamma1, Ng, Rabi):
    """∂P/∂t = −γ₁·(P+dP) + Ng·(E+dE) + iΩ·(S+dS)"""
    return -gamma1 * (P + dP) + Ng * (E + dE) + 1j * Rabi * (S + dS)

@njit(fastmath=True)
def rhs_S(P, dP, S, dS, gamma2, Rabi):
    """∂S/∂t = −γ₂·(S+dS) + iΩ*·(P+dP)"""
    return -gamma2 * (S + dS) + 1j * np.conjugate(Rabi) * (P + dP)


@njit(fastmath=True)
def rk4_step(E, P, S, Ng, c_light, gamma1, gamma2, Rabi, dt, dz):
    """
    One 4th-order Runge-Kutta step for all three fields simultaneously.
    Returns (E_new, P_new, S_new) at time t + dt.
    """
    # Stage 1 — slopes at t
    k1_E = rhs_E(E, 0, P, 0, Ng, c_light, dz)
    k1_P = rhs_P(E, 0, P, 0, S, 0, gamma1, Ng, Rabi)
    k1_S = rhs_S(P, 0, S, 0, gamma2, Rabi)

    # Stage 2 — slopes at t + dt/2 (using stage-1 prediction)
    k2_E = rhs_E(E, (dt/2)*k1_E, P, (dt/2)*k1_P, Ng, c_light, dz)
    k2_P = rhs_P(E, (dt/2)*k1_E, P, (dt/2)*k1_P, S, (dt/2)*k1_S, gamma1, Ng, Rabi)
    k2_S = rhs_S(P, (dt/2)*k1_P, S, (dt/2)*k1_S, gamma2, Rabi)

    # Stage 3 — slopes at t + dt/2 (using stage-2 prediction)
    k3_E = rhs_E(E, (dt/2)*k2_E, P, (dt/2)*k2_P, Ng, c_light, dz)
    k3_P = rhs_P(E, (dt/2)*k2_E, P, (dt/2)*k2_P, S, (dt/2)*k2_S, gamma1, Ng, Rabi)
    k3_S = rhs_S(P, (dt/2)*k2_P, S, (dt/2)*k2_S, gamma2, Rabi)

    # Stage 4 — slopes at t + dt (using stage-3 prediction)
    k4_E = rhs_E(E, dt*k3_E, P, dt*k3_P, Ng, c_light, dz)
    k4_P = rhs_P(E, dt*k3_E, P, dt*k3_P, S, dt*k3_S, gamma1, Ng, Rabi)
    k4_S = rhs_S(P, dt*k3_P, S, dt*k3_S, gamma2, Rabi)

    # Weighted sum (standard RK4 weights)
    E_new = E + (dt / 6) * (k1_E + 2*k2_E + 2*k3_E + k4_E)
    P_new = P + (dt / 6) * (k1_P + 2*k2_P + 2*k3_P + k4_P)
    S_new = S + (dt / 6) * (k1_S + 2*k2_S + 2*k3_S + k4_S)
    return E_new, P_new, S_new


In [ ]:
@njit(fastmath=True)
def run_simulation(E, P, S, z, dt, Ng, rabi_profile,
                   gamma1, gamma2, run_time, c_light,
                   size, dz, n_cut_frac, interval):
    """
    Time-march the Maxwell-Bloch system and collect periodic snapshots.

    Stores |E|, |P|, |S| (complex magnitudes) rather than real parts.
    Reason: during EIT storage the spin-wave S is predominantly imaginary
    (its phase is set by the Rabi field), so S.real ≈ 0 even when |S| is
    large. Plotting S.real² would show a flat zero during storage and a
    spurious spike at retrieval when the phase suddenly rotates.
    Using |S| gives the physically meaningful occupation of the spin wave.
    """
    n_z      = len(z)
    Es       = np.zeros(size, dtype=np.float32)   # stores |E|
    Ps       = np.zeros(size, dtype=np.float32)   # stores |P|
    Ss       = np.zeros(size, dtype=np.float32)   # stores |S|
    Zs       = np.zeros(size, dtype=np.float32)
    ts       = np.zeros(size[0], dtype=np.float64)
    cut_idx  = int(n_z * n_cut_frac)
    snap_idx = 0

    n_steps = int(run_time / dt)
    for step in range(n_steps):
        Rabi = rabi_profile[step]
        E, P, S = rk4_step(E, P, S, Ng, c_light, gamma1, gamma2, Rabi, dt, dz)

        # Absorbing left boundary
        E[:cut_idx] = 0.0 + 0.0j

        if step % interval == 0:
            # Store magnitudes — physically correct for all three fields
            for i in range(n_z):
                Es[snap_idx, i] = abs(E[i])
                Ps[snap_idx, i] = abs(P[i])
                Ss[snap_idx, i] = abs(S[i])
                Zs[snap_idx, i] = z[i].real
            ts[snap_idx] = step * dt
            snap_idx += 1

    return Es, Ps, Ss, Zs, ts


## Run the simulation


In [ ]:
E_init = signal.copy()
P_init = np.zeros_like(E_init)
S_init = np.zeros_like(E_init)

# ── Snapshot interval ─────────────────────────────────────────────────────────
# interval = 5   → very dense (good for slow animations, bad for memory)
# interval = 50  → balanced
# interval = 200 → minimal; enough for a clean animation at typical run_times
#
# Rule of thumb: aim for 500–2000 snapshots total.
# If run_time is large (>500 ns), increase interval accordingly.
interval   = max(5, int(run_time / dt / 1500))   # auto-target ~1500 snapshots
n_cut_frac = 0.10    # left 10 % of the grid acts as an absorbing boundary

MAX_MEMORY_MB = 1500  # hard stop — raise if you have more RAM available

n_steps     = int(run_time / dt)
n_snapshots = n_steps // interval + 1
size        = (n_snapshots, len(z))

# float32 storage (4 bytes) instead of complex128 (16 bytes)
mem_mb = 3 * n_snapshots * len(z) * 4 / 1e6

print(f"run_time             : {run_time:.1f} ns")
print(f"Grid points          : {len(z)}")
print(f"Time steps           : {n_steps}")
print(f"Interval             : {interval}  (one snapshot per {interval} steps)")
print(f"Snapshots            : {n_snapshots}")
print(f"Estimated memory     : {mem_mb:.1f} MB  (float32, 3 arrays)")

if mem_mb > MAX_MEMORY_MB:
    raise MemoryError(
        f"\n  Would need {mem_mb:.0f} MB > {MAX_MEMORY_MB} MB limit.\n"
        f"  Options:\n"
        f"    • Increase interval to ~{int(3 * n_steps * len(z) * 4 / (MAX_MEMORY_MB * 1e6))+1}\n"
        f"    • Reduce storage_time (currently {storage_time} ns)\n"
        f"    • Reduce max_amp_rabi to speed up vg (less slow-down factor)\n"
        f"    • Increase dz (currently {dz} mm) to shrink the grid"
    )

print("\nStarting simulation…")
t0 = time.time()
Es, Ps, Ss, Zs, ts = run_simulation(
    E_init, P_init, S_init, z, dt, Ng,
    rabi_profile, gamma1, gamma2,
    run_time, c,
    size, dz, n_cut_frac, interval
)
elapsed = time.time() - t0
print(f"Completed in {elapsed:.2f} s  ({n_steps/elapsed:.0f} steps/s)")

mask = ~(Es == 0).all(axis=1)
Es, Ps, Ss, Zs, ts = Es[mask], Ps[mask], Ss[mask], Zs[mask], ts[mask]
print(f"Valid snapshots: {Es.shape[0]}  ×  {Es.shape[1]} grid points")


## Physical metrics

Computes time delay, optical depth (OD), **storage efficiency**, and
the 1/e coherence decay time from the simulation output.


In [ ]:
# ── Time delay ────────────────────────────────────────────────────────────────
# Compare the time of peak E at the medium entrance vs exit,
# subtract the free-space transit time to isolate the EIT-induced delay.
entry_time          = ts[np.argmax(Es[:, enter_idx])]
exit_time           = ts[np.argmax(Es[:, exit_idx])]
free_space_transit  = (exit_ - enter) / c
time_delay          = exit_time - entry_time - free_space_transit

# ── Optical depth (dimensionless) ─────────────────────────────────────────────
# OD = g²·N·L / (c·γ_eg) = <|Ng|²> · L / (c · γ_eg)
# (|Ng| = g√N; OD drives single-pass absorption outside the EIT window)
OD = float(
    np.mean(np.abs(Ng[enter_idx:exit_idx])**2) * (exit_ - enter) / (c * gamma)
)
print(f"mean(np.abs(Ng[enter_idx:exit_idx])**2) = {np.mean(np.abs(Ng[enter_idx:exit_idx])**2):.4f}")
print(f"exit_idx = {exit_idx}, enter_idx = {enter_idx}, gamma = {gamma:.4f}")
print(f"exit_ - enter = {exit_ - enter:.2f} mm")
# ── Peak amplitude over time ──────────────────────────────────────────────────
peak_vs_time = np.max(Es, axis=1)

# ── Storage efficiency (energy ratio) ────────────────────────────────────────
# η = ∫|E_out(t)|² dt / ∫|E_in(t)|² dt
# Probe measured just inside the medium entrance (enter_idx) and exit (exit_idx).
energy_in  = np.trapezoid(Es[:, enter_idx    ]**2, ts)
energy_out = np.trapezoid(Es[:, exit_idx - 1 ]**2, ts)
storage_efficiency = float(energy_out / energy_in) if energy_in > 0 else 0.0

# Peak-based efficiency (simpler estimate, noisier for short pulses)
peak_in  = float(np.max(np.abs(Es[:, enter_idx    ])))
peak_out = float(np.max(np.abs(Es[:, exit_idx - 1 ])))
peak_efficiency = (peak_out / peak_in) ** 2 if peak_in > 0 else 0.0

# ── 1/e coherence decay time ─────────────────────────────────────────────────
# Track the E field at the z-point with largest amplitude at mid-storage,
# then find when it drops to 1/e of its peak value.
mid_store_snap = int(np.argmin(rabi_profile) / interval)
z_peak_idx     = int(np.argmax(np.abs(Es[mid_store_snap, :])))
E_at_peak_z    = Es[:, z_peak_idx]

i_peak         = int(np.argmax(E_at_peak_z))
i_store_end    = int(np.argmin(np.abs(ts - (ts[i_peak] + storage_time))))
E_slice        = E_at_peak_z[i_peak : i_store_end]
E_slice_norm   = E_slice / E_slice[0] if len(E_slice) > 0 and E_slice[0] != 0 else E_slice
i_1e           = int(np.argmin(np.abs(E_slice_norm - 1 / np.e)))
decay_time     = ts[i_peak + i_1e] - ts[i_peak]

# ── Summary ───────────────────────────────────────────────────────────────────
sep = "─" * 46
print(sep)
print(f"  Entry time             : {entry_time:.3e} ns")
print(f"  Exit time              : {exit_time:.3e} ns")
print(f"  Free-space transit     : {free_space_transit:.3e} ns")
print(f"  EIT time delay         : {time_delay:.3e} ns")
print(sep)
print(f"  Optical depth (OD)     : {OD:.4f}   [dimensionless]")
print(f"  1/e decay time         : {decay_time:.3e} ns")
print(sep)
print(f"  Storage efficiency η   : {storage_efficiency*100:.2f} %  (energy ratio)")
print(f"  Peak efficiency        : {peak_efficiency*100:.2f} %  (peak² ratio)")
print(f"  Minimum peak |E|       : {float(np.min(peak_vs_time)):.4g}")
print(sep)


## Diagnostic plots

Field intensities at three characteristic positions: just before the medium
(input), at the midpoint, and just after the medium (output).
The dashed overlay shows the Rabi control profile Ω(t).


In [ ]:
# ── z-positions for time-domain traces ───────────────────────────────────────
# z₁: just before the medium entrance (Ng=0 here → P and S always zero, correct)
# z₂: midpoint INSIDE the medium     (Ng≠0  → P and S are driven here)
# z₃: just inside the medium exit    (exit_idx-1, still Ng≠0)
#
# Using exit_idx for P/S is wrong: Ng[exit_idx:]=0 so P/S are never driven at
# exit_idx — they just exponentially decay there, giving a misleading flat zero.
z1 = enter_idx - 2          # before the medium  (E input reference)
z2 = (enter_idx + exit_idx) // 2   # midpoint inside medium
z3 = exit_idx - 1           # last driven point inside medium (P/S); E exits here

z_positions = {'Input  (z₁)': z1, 'Midpoint (z₂)': z2, 'Output (z₃)': z3}
colors = ['steelblue', 'tomato', 'darkorchid']
rabi_t = np.arange(0, run_time, dt)

# ── Storage window in time ────────────────────────────────────────────────────
# Find where rabi_profile drops below 5% of peak (= "inside the Ω well")
thresh = max_amp_rabi * 0.05
store_mask = rabi_profile < thresh
if store_mask.any():
    t_well_start = float(rabi_t[np.argmax(store_mask)])
    t_well_end   = float(rabi_t[len(rabi_t) - np.argmax(store_mask[::-1]) - 1])
else:
    t_well_start, t_well_end = 0, run_time

plt.style.use('ggplot')

# ═══════════════════════════════════════════════════════════════════════════════
# Panel 1: E field (full scale — E is large and well-behaved)
# ═══════════════════════════════════════════════════════════════════════════════
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (fname, farr) in zip(axes,
        [('E  (probe field)', Es), ('P  (polarization)', Ps), ('S  (spin wave)', Ss)]):

    for (pname, z_idx), color in zip(z_positions.items(), colors):
        ax.plot(ts, farr[:, z_idx]**2, lw=1.2, color=color, label=pname)

    # Shade the Ω=0 storage window
    ax.axvspan(t_well_start, t_well_end, alpha=0.07, color='navy', label='Storage (Ω≈0)')

    # Rabi overlay — scale it to the plot's own y-range for readability
    ax2 = ax.twinx()
    ax2.plot(rabi_t, rabi_profile, lw=0.8, color='black', linestyle='--', alpha=0.35)
    ax2.set_ylim(0, max_amp_rabi * 4)   # push Rabi line to top 25% of axes
    ax2.set_yticks([])

    ax.set_title(fname)
    ax.set_xlabel('time  [ns]')
    ax.set_ylabel('|field|²  [arb.]')

axes[0].legend(fontsize=8, loc='upper right')
plt.suptitle(
    'Field intensities — blue shading = Ω≈0 storage window\n'
    'P and S are only driven inside the medium (z₂ is midpoint; z₃ is last driven point)',
    fontsize=10)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# Panel 2: P and S zoomed into the storage window — reveals stored spin wave
# ═══════════════════════════════════════════════════════════════════════════════
# Clip the y-axis to the 98th percentile INSIDE the storage window to show
# the stored S without the retrieval spike dominating the scale.
t_mask = (ts >= t_well_start) & (ts <= t_well_end)

fig, axes2 = plt.subplots(1, 2, figsize=(13, 4))

for ax, (fname, farr) in zip(axes2, [('P  (polarization)', Ps), ('S  (spin wave)', Ss)]):
    storage_vals = farr[t_mask, z2]**2
    y_cap = np.percentile(storage_vals, 98) * 3 if storage_vals.max() > 0 else 1.0
    y_cap = max(y_cap, 1e-9)

    for (pname, z_idx), color in zip(z_positions.items(), colors):
        ax.plot(ts, farr[:, z_idx]**2, lw=1.2, color=color, label=pname)

    ax.axvspan(t_well_start, t_well_end, alpha=0.10, color='navy')
    ax.set_ylim(0, y_cap)          # cap removes retrieval spike → reveals stored field
    ax.set_title(f'{fname} — zoomed to storage scale\n'
                 f'(retrieval spike clipped; y-cap = {y_cap:.2e})')
    ax.set_xlabel('time  [ns]')
    ax.set_ylabel('|field|²  [arb.]')
    ax.legend(fontsize=8)

plt.suptitle('P and S during the storage window — S should be nonzero here if storage works',
             fontsize=10)
plt.tight_layout()
plt.show()

# ═══════════════════════════════════════════════════════════════════════════════
# Panel 3: Spatial profiles — most intuitive view of EIT storage
# Shows WHERE the excitation is at each phase
# ═══════════════════════════════════════════════════════════════════════════════
snap_times = {
    'Just before storage':  t_well_start * 0.95,
    'Mid-storage':         (t_well_start + t_well_end) / 2,
    'Just before retrieval': t_well_end   * 1.01,
}
fig, axes3 = plt.subplots(1, 3, figsize=(16, 4), sharey=False)

for ax, (label, t_target) in zip(axes3, snap_times.items()):
    idx = int(np.argmin(np.abs(ts - t_target)))
    ax.fill_between([enter, exit_], 0, max(Es[idx].max(), Ss[idx].max()) * 1.1,
                    alpha=0.07, color='green', label='Medium' if ax == axes3[0] else '')
    ax.plot(z_real, Es[idx], color='steelblue', lw=1.5, label='|E|')
    ax.plot(z_real, Ss[idx], color='tomato',    lw=1.5, label='|S|')
    ax.plot(z_real, Ps[idx] * 20, color='black', lw=0.9,
            linestyle='--', label='|P| × 20')   # scale P up so it's visible
    ax.axvline(enter, color='green', lw=0.8, linestyle=':')
    ax.axvline(exit_, color='green', lw=0.8, linestyle=':')
    ax.set_xlabel('z  [mm]')
    ax.set_ylabel('|field|  [arb.]')
    ax.set_title(f'{label}\n(t = {ts[idx]:.1f} ns,  Ω = {rabi_profile[min(int(ts[idx]/dt), len(rabi_profile)-1)]:.1f} GHz)')
    if ax == axes3[0]:
        ax.legend(fontsize=8)

plt.suptitle('Spatial field profiles — green shading = medium (the "well")\n'
             'This is the clearest view: E→0, S→nonzero during storage',
             fontsize=10)
plt.tight_layout()
plt.show()


## 3-D surface plot  E(z, t)


In [ ]:
n_snap, n_z = Zs.shape
T_grid      = np.tile(ts, (n_z, 1)).T   # broadcast ts → (n_snap, n_z)

fig = plt.figure(figsize=(10, 6))
ax  = fig.add_subplot(111, projection='3d')
ax.plot_surface(T_grid, Zs.real, Es.real, cmap='viridis',
                linewidth=0, antialiased=False, alpha=0.85)
ax.set_xlabel('time  [ns]')
ax.set_ylabel('z  [mm]')
ax.set_zlabel('E  [arb.]')
ax.set_title('Probe field E(z, t)')
plt.tight_layout()
plt.show()


## Inline animation  — E(z, t) propagation


In [ ]:
anim_speed = 2   # frames to skip per render step  (higher = faster playback)

plt.close('all')
fig, ax = plt.subplots(figsize=(9, 4))
ax.set_xlim(z_real.min(), z_real.max())
ax.set_ylim(float(Es.min()), float(Es.max()) * 1.05)
ax.axvline(enter, color='crimson', lw=1.2, label='Medium boundary')
ax.axvline(exit_, color='crimson', lw=1.2)
ax.set_xlabel('z  [mm]')
ax.set_ylabel('E  [arb.]')
ax.set_title('EIT probe-field propagation')
ax.legend(loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

(line,) = ax.plot([], [], lw=1.5, color='steelblue')

def _init():
    line.set_data([], [])
    return (line,)

def _animate(frame):
    snap = frame * anim_speed
    line.set_data(Zs[snap].real, Es[snap].real)
    return (line,)

anim = FuncAnimation(
    fig, _animate, init_func=_init,
    frames=len(Es) // anim_speed,
    interval=20, blit=False
)
anim


## MP4 export  *(optional — requires ffmpeg)*

Set `SAVE_MP4 = True` and ensure `ffmpeg` is on your PATH  
(or set `plt.rcParams['animation.ffmpeg_path']` to the full binary path).


In [ ]:
SAVE_MP4 = True   # ← flip to True to export

if SAVE_MP4:
    # Uncomment and set your ffmpeg path if it is not on PATH:
    # plt.rcParams['animation.ffmpeg_path'] = r'C:\path\to\ffmpeg.exe'

    anim_speed_mp4 = 5
    fps            = 10
    output_file    = f'eit3_{int(storage_time)}ns.mp4'

    # Normalise each field to [−1, 1] for a clean side-by-side comparison
    def _norm(arr):
        m = np.max(np.abs(arr))
        return arr / m if m > 0 else arr

    Es_n = _norm(Es);  Ps_n = _norm(Ps);  Ss_n = _norm(Ss)

    writer  = animation.FFMpegWriter(fps=fps, metadata=dict(title='EIT simulation'))
    fig_mp4, ax_mp4 = plt.subplots(figsize=(10, 4))

    with writer.saving(fig_mp4, output_file, dpi=100):
        for i in range(len(Es_n) // anim_speed_mp4):
            idx = i * anim_speed_mp4
            ax_mp4.cla()
            ax_mp4.plot(Es_n[idx], color='steelblue', lw=1.2, label='E')
            ax_mp4.plot(Ss_n[idx], color='tomato',    lw=1.2, label='S')
            ax_mp4.plot(Ps_n[idx], color='black',     lw=0.8, label='P')
            ax_mp4.set_ylim(-1.1, 1.1)
            ax_mp4.set_title('EIT — E (blue), S (red), P (black)')
            if i == 0:
                ax_mp4.legend()
            writer.grab_frame()

    plt.close(fig_mp4)
    print(f"Saved '{output_file}'")


## Save simulation results


In [ ]:
# Save all field snapshots and the z/t axes for later analysis.
prefix = f'{int(storage_time)}ns'
np.save(f'{prefix}_Es.npy', Es)
np.save(f'{prefix}_Ps.npy', Ps)
np.save(f'{prefix}_Ss.npy', Ss)
np.save(f'{prefix}_ts.npy', ts)
np.save(f'{prefix}_z.npy',  Zs[0].real)   # z-axis is identical for every snapshot

total_mb = (Es.nbytes + Ps.nbytes + Ss.nbytes + ts.nbytes) / 1e6
print(f"Saved  {prefix}_{{Es, Ps, Ss, ts, z}}.npy   —   {total_mb:.1f} MB total")
